# step5 — K-Means 최종 마무리 (k 확정 -> 라벨링 -> 프로파일링)

`step4_kmeans_clustering.py`에서 나온 k별 inertia/silhouette 결과를 이어받아,
여기서는 최종 k를 사람이 직접 비교/판단해서 확정하고 결과를 저장한다.

흐름:
1. step4 산출물(`kmeans_k_selection.csv`) 불러와 참고
2. 후보 k(2/3/4)를 PCA 2D로 나란히 비교
3. 최종 k로 재학습 + 클러스터 라벨 저장
4. 클러스터별 프로파일 확인 (해석/네이밍용)
5. 최종 PCA 시각화 저장
6. 다른 팀원(계층적/DBSCAN/GNN)과 비교할 지표 저장

이 단계는 "결과가 나오면 그래프 보고 다음 셀에서 k를 바꿔 다시 돌리는" 식의 반복 판단이 필요해서
스크립트보다 노트북이 더 적합함.

## 0. 환경 설정 및 데이터 로드

In [1]:
from pathlib import Path
import json as jsonlib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

# 한글 폰트 설정 (제목/라벨 깨짐 방지)
for _font_name in ("AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
    if _font_name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _font_name
        break
plt.rcParams["axes.unicode_minus"] = False

In [2]:
def _find_repo_root() -> Path:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() or (candidate / "data").is_dir():
            return candidate
    return start


ROOT = _find_repo_root()
INTERIM_DIR = ROOT / "data" / "processed" / "interim"
OUTPUT_DIR = ROOT / "data" / "processed" / "output"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# step3/step4와 동일한 피처 (투수-시즌 단위 평균 프로필)
FEATURE_COLS = [
    "release_speed",
    "ivb_ft",
    "hb_ft",
    "arm_angle",
    "release_pos_x",
    "release_pos_z",
    "release_extension",
]

RANDOM_STATE = 42

profile = pd.read_pickle(INTERIM_DIR / "fastball_profile_table.pkl")
print(f"투수-시즌 행 수: {len(profile):,}")
profile.head()

투수-시즌 행 수: 1,860


,pitcher,player_name,game_year,primary_fastball_type,release_speed,ivb_ft,hb_ft,arm_angle,release_pos_x,release_pos_z,release_extension
0,425794,"Wainwright, Adam",2021,SI,89.057643,11.296241,11.720254,44.741190,-1.206838,6.214632,6.564177
1,425794,"Wainwright, Adam",2022,SI,88.566476,11.014019,11.805261,44.235126,-1.118707,6.228341,6.508352
2,425794,"Wainwright, Adam",2023,SI,86.874952,9.813259,11.553840,43.350478,-1.256577,6.134436,6.551625
3,425844,"Greinke, Zack",2021,FF,88.936816,14.613054,1.149525,48.838086,-1.223340,6.406416,5.991016
4,425844,"Greinke, Zack",2022,FF,89.130657,14.118650,1.813353,44.005596,-1.688309,6.254672,5.927372


In [3]:
k_selection_path = OUTPUT_DIR / "kmeans_k_selection.csv"
if k_selection_path.exists():
    k_table = pd.read_csv(k_selection_path)
    print("[step4 산출물] k별 inertia / silhouette")
    display(k_table)
else:
    k_table = None
    print(f"{k_selection_path} 없음 -> step4를 먼저 실행하거나 이 표 없이 진행")

[step4 산출물] k별 inertia / silhouette


,k,inertia,silhouette
0,2,10210.297804,0.249043
1,3,8983.571793,0.172768
2,4,8004.273841,0.185866
3,5,7166.909987,0.186180
4,6,6554.941309,0.186343
5,7,6076.899381,0.173470
6,8,5678.476025,0.177819
7,9,5388.818487,0.176535
8,10,5122.323775,0.172688


In [4]:
X = profile[FEATURE_COLS].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 1. 후보 k 비교 (PCA 2D 나란히)

k=2/3/4를 나란히 그려서 실제로 얼마나 다른 구조인지 비교한다.
- k=2: silhouette 최고점이지만 큰 덩어리를 둘로 가른 것에 가까움
- k=3: 팔각도 낮은 그룹(싱커/컷터 계열)이 별도 클러스터로 분리됨 — 해석 가능한 구조
- k=4: 왼쪽 끝 이상치 소수 그룹이 별도로 분리됨 — 참고할 만하지만 표본이 매우 적음

**이 셀 결과를 보고 아래 `FINAL_K` 값을 정한다.**

In [5]:
compare_ks_list = [2, 3, 4]
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_.sum()

fig, axes = plt.subplots(1, len(compare_ks_list), figsize=(6 * len(compare_ks_list), 5.5))
for ax, k in zip(axes, compare_ks_list):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_tmp = km.fit_predict(X_scaled)
    sil_tmp = silhouette_score(X_scaled, labels_tmp)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_tmp, cmap="tab10", s=20, alpha=0.8)
    ax.set_title(f"k={k} (silhouette={sil_tmp:.3f})")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    print(f"k={k}: silhouette={sil_tmp:.4f}, 클러스터 크기={pd.Series(labels_tmp).value_counts().sort_index().to_dict()}")

fig.suptitle(f"k 비교 (PCA 2D, 설명된 분산 {explained:.1%})")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_k_comparison.png", dpi=150)
plt.show()

k=2: silhouette=0.2490, 클러스터 크기={0: 1289, 1: 571}
k=3: silhouette=0.1728, 클러스터 크기={0: 558, 1: 485, 2: 817}
k=4: silhouette=0.1859, 클러스터 크기={0: 509, 1: 811, 2: 28, 3: 512}


/var/folders/63/blb90k956gg_3ngmcpxm_0_c0000gn/T/ipykernel_18245/121167180.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. 최종 k 확정 및 재학습

위 비교 결과를 보고 `FINAL_K`를 정한다.
(지금까지 판단 근거: k=3에서 팔각도-무브먼트 관계가 뚜렷하게 해석되어 k=3을 채택함— 자세한 근거는 3절 프로파일링 참고)

In [6]:
FINAL_K = 3  # <- 1절 결과 보고 필요하면 여기 값을 바꿔서 이 셀부터 다시 실행

final_km = KMeans(n_clusters=FINAL_K, n_init=10, random_state=RANDOM_STATE)
labels = final_km.fit_predict(X_scaled)
sil_final = silhouette_score(X_scaled, labels)
sample_sil = silhouette_samples(X_scaled, labels)

profile["cluster"] = labels
profile["silhouette_sample"] = sample_sil

print(f"최종 k={FINAL_K}, silhouette={sil_final:.4f}")
profile["cluster"].value_counts().sort_index()

최종 k=3, silhouette=0.1728


cluster
0    558
1    485
2    817
Name: count, dtype: int64

## 3. 클러스터별 프로파일링 (해석/네이밍용)

In [7]:
cluster_profile = profile.groupby("cluster")[FEATURE_COLS].agg(["mean", "std"]).round(2)
cluster_profile.to_csv(OUTPUT_DIR / "kmeans_cluster_profile.csv")
cluster_profile

release_speed       ivb_ft        hb_ft       arm_angle         \
                 mean   std   mean   std   mean   std      mean    std   
cluster                                                                  
0               92.09  2.38  14.08  3.85   5.56  5.38     45.91   8.56   
1               93.52  2.69   7.89  4.89  14.54  3.33     25.49  14.81   
2               95.32  1.92  16.87  2.24   7.76  3.61     42.73   8.76   

        release_pos_x       release_pos_z       release_extension        
                 mean   std          mean   std              mean   std  
cluster                                                                  
0                0.24  1.75          6.08  0.36              6.18  0.36  
1               -0.67  2.31          5.35  0.71              6.38  0.44  
2               -1.43  1.22          5.89  0.36              6.63  0.35

각 클러스터의 구속/IVB/HB/팔각도 평균을 보고 이름을 붙인다.
예) 팔각도 낮고 HB 크고 IVB 작음 -> 싱커/컷터 계열 / 구속·IVB 모두 높음 -> 엘리트급 라이징 포심 등

## 4. 최종 PCA 시각화

In [8]:
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="tab10", s=25, alpha=0.8)
ax.set_title(f"K-Means Clusters (k={FINAL_K}) — PCA 2D\n설명된 분산: {explained:.1%}")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.colorbar(scatter, label="cluster")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_pca_clusters.png", dpi=150)
plt.show()

/var/folders/63/blb90k956gg_3ngmcpxm_0_c0000gn/T/ipykernel_18245/1294093845.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. 다른 팀원(계층적/DBSCAN/GNN)과 비교할 지표 저장

In [9]:
sizes = profile["cluster"].value_counts().sort_index()
metrics = {
    "method": "KMeans",
    "unit": "pitcher-season (primary fastball)",
    "n_clusters": int(FINAL_K),
    "silhouette_score": round(float(sil_final), 4),
    "cluster_sizes": {int(c): int(n) for c, n in sizes.items()},
    "noise_points": 0,
    "notes": (
        "K-Means는 구형(spherical) 클러스터를 가정하고 k를 사전에 지정해야 함. "
        "DBSCAN과 달리 이상치를 별도 노이즈로 분리하지 않고 모든 투수-시즌을 "
        "클러스터에 할당함. 계층적/GNN 결과와 클러스터 수·크기 비교 시 참고."
    ),
}

with open(OUTPUT_DIR / "kmeans_comparison_metrics.json", "w", encoding="utf-8") as f:
    jsonlib.dump(metrics, f, ensure_ascii=False, indent=2)

metrics

{'method': 'KMeans',
 'unit': 'pitcher-season (primary fastball)',
 'n_clusters': 3,
 'silhouette_score': 0.1728,
 'cluster_sizes': {0: 558, 1: 485, 2: 817},
 'noise_points': 0,
 'notes': 'K-Means는 구형(spherical) 클러스터를 가정하고 k를 사전에 지정해야 함. DBSCAN과 달리 이상치를 별도 노이즈로 분리하지 않고 모든 투수-시즌을 클러스터에 할당함. 계층적/GNN 결과와 클러스터 수·크기 비교 시 참고.'}

## 6. 최종 결과 저장

In [10]:
out_path = OUTPUT_DIR / "fastball_clusters_final.csv"
profile.to_csv(out_path, index=False)
print(f"저장 완료: {out_path} ({len(profile):,}행)")

저장 완료: /Users/yoon-uijin/Documents/GitHub/9th-first-project-baseball-1/data/processed/output/fastball_clusters_final.csv (1,860행)


다음 단계: `step6_kmeans_conclusion.ipynb`에서 `primary_fastball_type`과 클러스터 교차표를 통해
위 해석(팔각도 ↔ 무브먼트 방향)을 검증하고 최종 결론을 정리한다.